In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

In [ ]:
%pip install faiss-cpu

Geração de embeddings e faiss temporários para verificação de duplicatas

In [ ]:
import pandas as pd
import numpy as np
import re
import faiss

from sentence_transformers import SentenceTransformer

input_file = (
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/banco_python_concatenado.parquet"
)

candidate_file = (
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/candidatos_duplicatas_semanticas.parquet"
)

df = pd.read_parquet(input_file)

df["id_original"] = np.arange(len(df))

print("=" * 80)
print("DETECÇÃO DE DUPLICATAS SEMÂNTICAS")
print("=" * 80)

print(f"Total de registros carregados: {len(df):,}")

required_columns = [
    "question",
    "answer",
    "source"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Colunas ausentes: {missing_columns}"
    )

initial_count = len(df)

df = (
    df
    .drop_duplicates(
        subset=["question", "answer"],
        keep="first"
    )
    .reset_index(drop=True)
)

removed_exact_duplicates = initial_count - len(df)

print("\n" + "=" * 80)
print("DUPLICATAS EXATAS")
print("=" * 80)

print(f"Removidas: {removed_exact_duplicates:,}")
print(f"Registros restantes: {len(df):,}")

print("\n" + "=" * 80)
print("CARREGANDO MODELO")
print("=" * 80)

model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

questions = (
    df["question"]
    .fillna("")
    .astype(str)
    .str.strip()
)

valid_mask = questions != ""

df = (
    df[valid_mask]
    .reset_index(drop=True)
)

questions = (
    questions[valid_mask]
    .reset_index(drop=True)
)

print(f"Perguntas válidas: {len(df):,}")

print("\n" + "=" * 80)
print("GERANDO EMBEDDINGS TEMPORÁRIOS")
print("=" * 80)

embeddings = model.encode(
    questions.tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.asarray(
    embeddings,
    dtype=np.float32
)

print("\nEmbeddings temporários:")
print(f"Shape: {embeddings.shape}")
print(f"Dimensão: {embeddings.shape[1]}")

embedding_dimension = embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(embedding_dimension)

faiss_index.add(embeddings)

print("\n" + "=" * 80)
print("ÍNDICE FAISS TEMPORÁRIO")
print("=" * 80)

print(
    f"Vetores no índice: {faiss_index.ntotal:,}"
)

k = 6

similarities, indices = faiss_index.search(
    embeddings,
    k
)

candidates = []

for i in range(len(df)):

    for position in range(1, k):

        j = int(indices[i, position])

        if j < 0:
            continue

        similarity = float(
            similarities[i, position]
        )

        candidates.append({

            "indice_1": i,
            "indice_2": j,

            "id_original_1": int(df.iloc[i]["id_original"]),
            "id_original_2": int(df.iloc[j]["id_original"]),

            "similaridade": similarity,

            "question_1": df.iloc[i]["question"],
            "question_2": df.iloc[j]["question"],

            "answer_1": df.iloc[i]["answer"],
            "answer_2": df.iloc[j]["answer"],

            "source_1": df.iloc[i]["source"],
            "source_2": df.iloc[j]["source"]

        })

df_candidates = pd.DataFrame(candidates)

similarity_threshold = 0.80

df_candidates = df_candidates[
    df_candidates["similaridade"] >= similarity_threshold
].copy()

df_candidates["par"] = df_candidates.apply(

    lambda row: tuple(
        sorted([
            int(row["indice_1"]),
            int(row["indice_2"])
        ])
    ),

    axis=1
)

df_candidates = (
    df_candidates
    .drop_duplicates(
        subset=["par"]
    )
    .drop(
        columns=["par"]
    )
)

df_candidates = (
    df_candidates
    .sort_values(
        by="similaridade",
        ascending=False
    )
    .reset_index(drop=True)
)


def normalize_text(text):

    if pd.isna(text):
        return ""

    text = str(text).strip().lower()

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text


def classify_duplicate(row):

    question_1 = normalize_text(
        row["question_1"]
    )

    question_2 = normalize_text(
        row["question_2"]
    )

    answer_1 = normalize_text(
        row["answer_1"]
    )

    answer_2 = normalize_text(
        row["answer_2"]
    )

    if question_1 == question_2:

        if answer_1 == answer_2:

            return "duplicata_exata"

        else:

            return "mesma_pergunta_respostas_diferentes"

    else:

        return "duplicata_semantica"


df_candidates["tipo_duplicata"] = (
    df_candidates.apply(
        classify_duplicate,
        axis=1
    )
)

print("\n" + "=" * 80)
print("CLASSIFICAÇÃO DOS CANDIDATOS")
print("=" * 80)

if len(df_candidates) > 0:

    print(
        df_candidates[
            "tipo_duplicata"
        ]
        .value_counts()
        .to_string()
    )

else:

    print("Nenhum candidato encontrado.")

df_candidates.to_parquet(
    candidate_file,
    index=False
)

print("\n" + "=" * 80)
print("RESULTADO")
print("=" * 80)

print(
    f"Candidatos a duplicatas semânticas: "
    f"{len(df_candidates):,}"
)

print(
    "\nArquivo salvo em:"
)

print(candidate_file)

print("\n" + "=" * 80)
print("DISTRIBUIÇÃO DAS SIMILARIDADES")
print("=" * 80)

if len(df_candidates) > 0:

    print(
        df_candidates[
            "similaridade"
        ]
        .describe()
        .to_string()
    )

else:

    print("Nenhum candidato encontrado.")

print("\n" + "=" * 80)
print("AMOSTRA DOS CANDIDATOS")
print("=" * 80)

for i, row in df_candidates.head(30).iterrows():

    print("\n" + "-" * 80)

    print(
        f"Similaridade: "
        f"{row['similaridade']:.4f}"
    )

    print(
        f"TIPO: "
        f"{row['tipo_duplicata']}"
    )

    print("\nPERGUNTA 1:")
    print(row["question_1"])

    print("\nPERGUNTA 2:")
    print(row["question_2"])

    print("\nRESPOSTA 1:")
    print(
        str(row["answer_1"])[:700]
    )

    print("\nRESPOSTA 2:")
    print(
        str(row["answer_2"])[:700]
    )

    print(
        f"\nSOURCE 1: "
        f"{row['source_1']}"
    )

    print(
        f"SOURCE 2: "
        f"{row['source_2']}"
    )

print("\n" + "=" * 80)
print("ETAPA CONCLUÍDA")
print("=" * 80)

print(
    "As embeddings geradas nesta etapa são temporárias."
)

print(
    "O FAISS criado nesta etapa também é temporário."
)

print(
    "Nenhuma duplicata semântica foi removida automaticamente."
)

print(
    "Os candidatos devem ser analisados antes da "
    "criação do banco definitivo."
)

Verificação dos candidatos a duplicatas

In [ ]:
import pandas as pd

file_candidates = (
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/candidatos_duplicatas_semanticas.parquet"
)

df_dup = pd.read_parquet(file_candidates)

print("Total de pares:", len(df_dup))
print("\nColunas:")
print(df_dup.columns.tolist())

display(df_dup.head(10))

Verificação dos candidatos a duplicatas exatas

In [ ]:
df_exacts = df_dup[
    df_dup["tipo_duplicata"] == "duplicata_exata"
].copy()

print("Duplicatas exatas:", len(df_exacts))

display(
    df_exacts[
        [
            "indice_1",
            "indice_2",
            "similaridade",
            "question_1",
            "question_2",
            "answer_1",
            "answer_2",
            "source_1",
            "source_2"
        ]
    ].head(30)
)

Verificação dos candidatos a duplicatas semânticas

In [ ]:
df_semantics = df_dup[
    df_dup["tipo_duplicata"] == "duplicata_semantica"
].copy()

df_semantics = df_semantics.sort_values(
    "similaridade",
    ascending=False
)

display(
    df_semantics[
        [
            "indice_1",
            "indice_2",
            "similaridade",
            "question_1",
            "question_2",
            "source_1",
            "source_2"
        ]
    ].head(50)
)

Verificação dos candidatos a duplicatas com mesma pergunta e respostas difernetes

In [ ]:
df_semantics = df_dup[
    df_dup["tipo_duplicata"] == "mesma_pergunta_respostas_diferentes"
].copy()

df_semantics = df_semantics.sort_values(
    "similaridade",
    ascending=False
)

display(
    df_semantics[
        [
            "indice_1",
            "indice_2",
            "similaridade",
            "question_1",
            "question_2",
            "answer_1",
            "answer_2",
            "source_1",
            "source_2"
        ]
    ].head(50)
)

Limpeza do banco.parquet em relação as duplicatas



In [ ]:
import pandas as pd


candidates_path = (
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/candidatos_duplicatas_semanticas.parquet"
)

input_path = (
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/banco_python_concatenado.parquet"
)

output_path = (
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/banco_python_finalizado.parquet"
)


df_candidates = pd.read_parquet(
    candidates_path
)

df_original = pd.read_parquet(
    input_path
)


print("=" * 80)
print("LIMPEZA DO BANCO PYTHON")
print("=" * 80)

print(
    f"Registros no arquivo principal: "
    f"{len(df_original):,}"
)

print("\n" + "=" * 80)
print("1. DUPLICATAS EXATAS")
print("=" * 80)


df_exact_duplicates = df_candidates[
    df_candidates["duplicate_type"]
    == "exact_duplicate"
].copy()


print(
    f"Duplicatas exatas encontradas: "
    f"{len(df_exact_duplicates):,}"
)


if len(df_exact_duplicates) > 0:

    ids_exact_duplicates = set(
        df_exact_duplicates["original_id_2"].astype(int)
    )

else:

    ids_exact_duplicates = set()


print(
    f"IDs de duplicatas exatas a remover: "
    f"{len(ids_exact_duplicates):,}"
)

print("\n" + "=" * 80)
print("2. MESMA PERGUNTA COM RESPOSTAS DIFERENTES")
print("=" * 80)


df_same_questions = df_candidates[
    df_candidates["duplicate_type"]
    == "same_question_different_answers"
].copy()


print(
    f"Casos encontrados: "
    f"{len(df_same_questions):,}"
)


def score_answer(answer, source):

    if pd.isna(answer):
        answer = ""

    answer = str(answer).strip()

    score = 0

    if answer:
        score += 1

    if len(answer) >= 100:
        score += 1

    if len(answer) >= 300:
        score += 1

    if len(answer) >= 600:
        score += 1

    if "```" in answer:
        score += 2

    if str(source).lower() == "python_docs":
        score += 3

    return score


def determine_removal(row):

    id_1 = int(row["original_id_1"])
    id_2 = int(row["original_id_2"])

    score_1 = row["score_1"]
    score_2 = row["score_2"]

    if score_1 >= score_2:

        return pd.Series({
            "id_to_keep": id_1,
            "id_to_remove": id_2,
            "retained_score": score_1,
            "removed_score": score_2
        })

    else:

        return pd.Series({
            "id_to_keep": id_2,
            "id_to_remove": id_1,
            "retained_score": score_2,
            "removed_score": score_1
        })


if len(df_same_questions) > 0:

    df_same_questions["score_1"] = (
        df_same_questions.apply(
            lambda row: score_answer(
                row["answer_1"],
                row["source_1"]
            ),
            axis=1
        )
    )

    df_same_questions["score_2"] = (
        df_same_questions.apply(
            lambda row: score_answer(
                row["answer_2"],
                row["source_2"]
            ),
            axis=1
        )
    )

    removal_decisions = df_same_questions.apply(
        determine_removal,
        axis=1
    )

    df_same_questions = pd.concat(
        [
            df_same_questions,
            removal_decisions
        ],
        axis=1
    )

    ids_same_questions = set(
        df_same_questions[
            "id_to_remove"
        ].astype(int)
    )

else:

    ids_same_questions = set()

    print(
        "Nenhum caso de mesma pergunta com "
        "respostas diferentes foi encontrado."
    )


print(
    f"IDs de mesmas perguntas a remover: "
    f"{len(ids_same_questions):,}"
)


print("\n" + "=" * 80)
print("3. CONSOLIDANDO REMOÇÕES")
print("=" * 80)


ids_to_remove = (
    ids_exact_duplicates
    | ids_same_questions
)


print(
    f"Duplicatas exatas: "
    f"{len(ids_exact_duplicates):,}"
)

print(
    f"Mesmas perguntas: "
    f"{len(ids_same_questions):,}"
)

print(
    f"IDs únicos para remover: "
    f"{len(ids_to_remove):,}"
)


invalid_ids = [
    index
    for index in ids_to_remove
    if index < 0 or index >= len(df_original)
]


if invalid_ids:

    raise ValueError(
        f"Foram encontrados "
        f"{len(invalid_ids)} IDs inválidos. "
        f"Exemplos: {invalid_ids[:10]}"
    )


print("\nTodos os IDs são válidos.")


df_cleaned = (
    df_original
    .drop(index=list(ids_to_remove))
    .reset_index(drop=True)
)


print("\n" + "=" * 80)
print("CONFERÊNCIA FINAL")
print("=" * 80)


print(
    f"Registros antes: "
    f"{len(df_original):,}"
)

print(
    f"Duplicatas exatas: "
    f"{len(ids_exact_duplicates):,}"
)

print(
    f"Mesmas perguntas: "
    f"{len(ids_same_questions):,}"
)

print(
    f"IDs únicos removidos: "
    f"{len(ids_to_remove):,}"
)

print(
    f"Registros depois: "
    f"{len(df_cleaned):,}"
)

print(
    f"Registros removidos: "
    f"{len(df_original) - len(df_cleaned):,}"
)


print("\n" + "=" * 80)
print("AMOSTRA — DUPLICATAS EXATAS")
print("=" * 80)


for _, row in df_exact_duplicates.head(10).iterrows():

    id_1 = int(row["original_id_1"])
    id_2 = int(row["original_id_2"])

    print("\n" + "-" * 80)

    print("ID 1:", id_1)

    print(
        "PERGUNTA:",
        df_original.iloc[id_1]["question"]
    )

    print(
        "RESPOSTA:",
        str(
            df_original.iloc[id_1]["answer"]
        )[:300]
    )

    print("\nID 2:", id_2)

    print(
        "PERGUNTA:",
        df_original.iloc[id_2]["question"]
    )

    print(
        "RESPOSTA:",
        str(
            df_original.iloc[id_2]["answer"]
        )[:300]
    )

    print(
        "\n>>> REMOVER:",
        id_2
    )


print("\n" + "=" * 80)
print("AMOSTRA — MESMAS PERGUNTAS")
print("=" * 80)


for _, row in df_same_questions.head(10).iterrows():

    id_1 = int(row["original_id_1"])
    id_2 = int(row["original_id_2"])

    print("\n" + "-" * 80)

    print("ID 1:", id_1)
    print("SOURCE 1:", row["source_1"])
    print("SCORE 1:", row["score_1"])

    print(
        "PERGUNTA 1:",
        df_original.iloc[id_1]["question"]
    )

    print(
        "RESPOSTA 1:",
        str(
            df_original.iloc[id_1]["answer"]
        )[:500]
    )

    print("\nID 2:", id_2)
    print("SOURCE 2:", row["source_2"])
    print("SCORE 2:", row["score_2"])

    print(
        "PERGUNTA 2:",
        df_original.iloc[id_2]["question"]
    )

    print(
        "RESPOSTA 2:",
        str(
            df_original.iloc[id_2]["answer"]
        )[:500]
    )

    print(
        "\n>>> MANTER:",
        row["id_to_keep"]
    )

    print(
        ">>> REMOVER:",
        row["id_to_remove"]
    )


df_cleaned.to_parquet(
    output_path,
    index=False
)


print("\n" + "=" * 80)
print("ARQUIVO FINAL SALVO")
print("=" * 80)

print(output_path)

Geração de embeddings com Sentence Transformers (sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2)

In [ ]:
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

CURATED_DIR = Path(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated"
)

EMBEDDINGS_DIR = CURATED_DIR / "embeddings"

EMBEDDINGS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

INPUT_FILE = CURATED_DIR / "banco_python_finalizado.parquet"

SAVE_BATCH_SIZE = 5000
ENCODE_BATCH_SIZE = 32

MODEL_NAME = (
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

print("=" * 80)
print("GERAÇÃO DAS EMBEDDINGS DEFINITIVAS")
print("=" * 80)

print("\nCarregando banco...")

database = pd.read_parquet(INPUT_FILE)

database = database.reset_index(drop=True)

print(f"Total de registros: {len(database):,}")

required_columns = [
    "question",
    "answer",
    "code",
    "source",
    "language"
]

missing_columns = [
    column
    for column in required_columns
    if column not in database.columns
]

if missing_columns:
    raise ValueError(
        f"Colunas ausentes no banco: {missing_columns}"
    )

questions = (
    database["question"]
    .fillna("")
    .astype(str)
    .str.strip()
)

empty_question_count = (
    questions == ""
).sum()

print(
    f"Perguntas vazias: {empty_question_count:,}"
)

if empty_question_count > 0:
    raise ValueError(
        "Existem perguntas vazias no banco definitivo."
    )

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    f"Dispositivo utilizado: {device}"
)

print("\nCarregando modelo de embeddings...")

model = SentenceTransformer(
    MODEL_NAME,
    device=device
)

print("Modelo carregado.")

print("\n" + "=" * 80)
print("CONFIGURAÇÃO")
print("=" * 80)

print(f"Modelo: {MODEL_NAME}")
print(f"Batch de salvamento: {SAVE_BATCH_SIZE}")
print(f"Batch do modelo: {ENCODE_BATCH_SIZE}")
print("Normalização: True")
print("Tipo dos vetores: float32")
print("Dimensão esperada: 384")

total_records = len(database)

for start in tqdm(
    range(0, total_records, SAVE_BATCH_SIZE),
    desc="Gerando embeddings"
):

    end = min(
        start + SAVE_BATCH_SIZE,
        total_records
    )

    output_file = (
        EMBEDDINGS_DIR /
        f"lote_{start:07d}.parquet"
    )

    if output_file.exists():

        print(
            f"\nLote {start:,}-{end:,} "
            "já existe. Pulando..."
        )

        continue

    batch = None
    texts = None
    embeddings = None

    try:

        batch = database.iloc[
            start:end
        ].copy()

        texts = (
            batch["question"]
            .fillna("")
            .astype(str)
            .str.strip()
            .tolist()
        )

        embeddings = model.encode(
            texts,
            batch_size=ENCODE_BATCH_SIZE,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False
        )

        embeddings = np.asarray(
            embeddings,
            dtype=np.float32
        )

        if embeddings.shape[0] != len(batch):

            raise ValueError(
                "Quantidade de embeddings diferente "
                "da quantidade de registros."
            )

        if embeddings.shape[1] != 384:

            raise ValueError(
                f"Dimensão inesperada: "
                f"{embeddings.shape[1]}"
            )

        if not np.isfinite(embeddings).all():

            raise ValueError(
                "Foram encontrados valores NaN "
                "ou infinitos nas embeddings."
            )

        embedding_norms = np.linalg.norm(
            embeddings,
            axis=1
        )

        if not np.allclose(
            embedding_norms,
            1.0,
            atol=1e-4
        ):

            raise ValueError(
                "As embeddings não estão "
                "normalizadas corretamente."
            )

        batch["embedding"] = [
            vector
            for vector in embeddings
        ]

        batch.to_parquet(
            output_file,
            index=False
        )

        print(
            f"\nLote salvo: "
            f"{start:,} → {end:,}"
        )

        print(
            f"Shape: {embeddings.shape}"
        )

        print(
            f"Dtype: {embeddings.dtype}"
        )

    except Exception as e:

        print(
            f"\nERRO NO LOTE "
            f"{start:,}: {e}"
        )

        raise

    finally:

        del batch
        del texts
        del embeddings

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("\n" + "=" * 80)
print("PROCESSAMENTO CONCLUÍDO")
print("=" * 80)

print("As embeddings definitivas foram geradas em lotes.")
print(f"Diretório: {EMBEDDINGS_DIR}")
print("Modelo: paraphrase-multilingual-MiniLM-L12-v2")
print("Dimensão: 384")
print("Tipo: float32")
print("Normalização: True")

Confirmação da geração de embeddings em lotes

In [ ]:
from pathlib import Path

EMBEDDINGS_DIR = Path(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/embeddings"
)

embedding_files = list(
    EMBEDDINGS_DIR.glob("*.parquet")
)

print("Arquivos encontrados:", len(embedding_files))

for file in embedding_files[:5]:
    print(file.name)

Validação de um lote

In [ ]:
import pandas as pd
from pathlib import Path

input_path = Path(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/embeddings/lote_0000000.parquet"
)

df = pd.read_parquet(input_path)

print(df.shape)
print(df.columns)

print("\nPrimeiro registro:")
print(df.iloc[0][["question", "answer"]])

print("\nDimensão do embedding:")
print(len(df.iloc[0]["embedding"]))

Verificação do assuntos e estrutura dos lotes

In [ ]:
import pandas as pd

files = [
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/embeddings/lote_0000000.parquet"
]

for file in files:
    print("=" * 100)
    print(f"Arquivo: {file}")

    df = pd.read_parquet(files)

    print("\nColunas:")
    print(df.columns.tolist())

    print("\nTipos:")
    print(df.dtypes)

    print("\nPrimeiras linhas:")

    print(df.head())

Validação em um lote de embeddings

In [ ]:
df = pd.read_parquet(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/embeddings/lote_0000000.parquet"
)

print(df["embedding"].iloc[0].dtype)
print(df["embedding"].iloc[0].shape)